# 07. 오차역전파

오차역전파(Backpropagation)는 손실에서 시작해 계산 과정을 거꾸로 따라가며 각 파라미터의 기울기를 계산하는 방법이다.

수학적으로는 합성함수의 연쇄법칙을 신경망 전체에 반복 적용한 것이다.


![](https://d.pr/i/4nYlr4+)

## 1. 순전파, 역전파, 파라미터 갱신

PyTorch 모델의 한 번의 학습 step은 다음 네 단계로 진행된다.

1. **순전파(Forward Propagation)**
   - 입력을 모델에 통과시켜 예측값을 계산한다.
   - PyTorch 코드: `logits = model(X)`

2. **손실 계산(Loss Calculation)**
   - 예측값과 정답의 차이를 하나의 loss로 계산한다.
   - PyTorch 코드: `loss = criterion(logits, y)`

3. **오차역전파(Backpropagation)**
   - 손실을 기준으로 각 파라미터의 기울기를 계산한다.
   - PyTorch 코드: `loss.backward()`
   - 이 단계에서는 가중치가 아직 변경되지 않는다.

4. **파라미터 갱신(Parameter Update)**
   - optimizer가 계산된 기울기를 이용해 가중치와 편향을 변경한다.
   - PyTorch 코드: `optimizer.step()`

역전파를 "오차를 뒤로 보내 가중치를 조정하는 과정"이라고 한 문장으로 설명하면 `backward()`와 `step()`이 하나의 작업처럼 들릴 수 있다. 정확히는 **역전파가 기울기를 계산하고 optimizer가 파라미터를 갱신한다.**

### 간단한 2층 신경망

입력 $x$, 은닉층 출력 $h$, 최종 예측 $\hat{y}$가 다음 순서로 계산된다고 하자.

$$
z_1 = w_1x+b_1, \qquad h=f(z_1)
$$

$$
z_2=w_2h+b_2, \qquad \hat{y}=z_2
$$

정답을 $t$라고 하고 제곱 오차를 사용하면 다음과 같다.

$$
L=(\hat{y}-t)^2
$$

학습의 목표는 $L$을 줄이도록 $w_1$, $b_1$, $w_2$, $b_2$를 갱신하는 것이다.

In [1]:
import torch


import torch.nn as nn


import torch.optim as optim


torch.manual_seed(42)


## 2. 연쇄법칙과 autograd

다음 합성함수를 생각해 보자.

$$
y=x^2, \qquad z=2y
$$

`x`가 변할 때 `z`의 변화율은 각 단계의 변화율을 곱해 계산한다.

$$
\frac{dz}{dx}=\frac{dz}{dy}\times\frac{dy}{dx}=2\times2x=4x
$$

`x=3`이면 기울기는 `12`이다. PyTorch는 순전파 중 계산 그래프를 기록하고 `backward()`에서 같은 계산을 자동으로 수행한다.

In [2]:
# 입력값 X
x_value = 3.0

# 도함수 ==> x=3 대입 시 기울기 12
manual_gradient = 2 * (2 * x_value)

# 위 수동계산
# 아래 자동계산

# 파이토치로 자동계산
x = torch.tensor(3.0, requires_grad=True)

y = x ** 2
z = 2 * y

z.backward()

print("수동 계산 기울기:", manual_gradient)
print("autograd 기울기:", x.grad.item())


수동 계산 기울기: 12.0
autograd 기울기: 12.0


두 기울기가 모두 `12`이면 autograd가 연쇄법칙을 올바르게 적용한 것이다.
신경망에서는 이 계산이 수많은 Tensor와 계층에 걸쳐 수행되지만 사용 방법은 동일하다.

- `requires_grad=True`: 해당 Tensor와 관련된 연산을 추적한다.
- `backward()`: 최종 결과에서 시작해 기울기를 계산한다.
- `.grad`: 계산된 기울기가 저장된다.

## 3. 신경망 수준의 연쇄법칙

출력층과 가까운 $w_2$의 기울기는 비교적 짧은 경로로 계산된다.

$$
\frac{\partial L}{\partial w_2}
=
\frac{\partial L}{\partial \hat{y}}
\frac{\partial \hat{y}}{\partial z_2}
\frac{\partial z_2}{\partial w_2}
$$

입력층과 가까운 $w_1$은 더 많은 중간 연산을 거치므로 경로에 있는 미분값을 모두 곱한다.

$$
\frac{\partial L}{\partial w_1}
=
\frac{\partial L}{\partial \hat{y}}
\frac{\partial \hat{y}}{\partial z_2}
\frac{\partial z_2}{\partial h}
\frac{\partial h}{\partial z_1}
\frac{\partial z_1}{\partial w_1}
$$

이처럼 역전파는 각 연산이 받은 기울기에 자신의 **국소 기울기(local gradient)** 를 곱해 앞쪽으로 전달한다.

### 자주 사용하는 핵심 도함수

| 함수 또는 조합 | 핵심 도함수 | 수업에서 볼 내용 |
| --- | --- | --- |
| MSE | $\frac{\partial L}{\partial \hat{y}}=2(\hat{y}-y)$ | 오차가 클수록 기울기도 커짐 |
| Sigmoid | $\sigma'(z)=\sigma(z)(1-\sigma(z))$ | 양 끝 구간에서 기울기가 작아짐 |
| ReLU | $f'(z)=1\;(z>0),\;0\;(z\leq0)$ | 음수 구간에서 기울기가 차단됨 |
| Sigmoid + BCE | $\frac{\partial L}{\partial z}=\sigma(z)-y$ | 이진분류 출력층의 결합 결과 |
| Softmax + Cross Entropy | $\frac{\partial L}{\partial z_i}=s_i-y_i$ | 다중분류 출력층의 결합 결과 |

위 식은 단일 샘플 중심의 핵심 형태이다. `reduction='mean'`으로 여러 샘플의 손실을 평균내면 batch 크기에 따른 평균 계수가 함께 적용된다.

다음 코드는 ReLU가 포함된 2층 신경망에서 수동으로 계산한 기울기와 autograd가 계산한 기울기를 비교한다.

In [3]:
x = torch.tensor(2.0)
target = torch.tensor(1.0)

# requires_grad=True -> 파이토치가 기억하게 만들어서 W, b를 사용하거나 업데이트 할 수 있게 함
# 임의로 작성한 첫 번째 선형 방정식의 W, b
w1 = torch.tensor(0.5, requires_grad=True)
b1 = torch.tensor(0.1, requires_grad=True)

# 임의로 작성한 두 번째 선형 방정식의 W, b
w2 = torch.tensor(0.8, requires_grad=True)
b2 = torch.tensor(0.2, requires_grad=True)

# 순전파 1단계 -> Linear(1,1) 계산 수행
z1 = w1 * x + b1

# 순전파 2단계 -> 활성화, 0 이하면 0, 양수이면 값 그대로
h = torch.relu(z1)

# 순전파 3단계 -> Linear(1,1) 계산 수행
prediction = w2 * h + b2

# 순전파 4단계 -> 오차 계산, 정답과 예측값의 차이를 loss로 만듦
loss = (prediction - target) ** 2


prediction_value = prediction.detach().item()
h_value = h.detach().item()
z1_value = z1.detach().item()
print(f"prediction: {prediction_value:.6f}, h: {h_value:.6f}, z1: {z1_value:.6f}")

# 역전파 1단계: 손실을 예측값으로 미분
dL_dprediction = 2 * (prediction_value - target.item())

# 역전파 2단계: 출력층 파라미터 w2, b2의 기울기 계산
manual_dw2 = dL_dprediction * h_value
manual_db2 = dL_dprediction

# 역전파 3단계: 기울기를 출력층에서 첫번째 층으로 전달
relu_gradient = 1.0 if z1_value > 0 else 0.0

# z1까지 전달되는 기울기는 경로에 있는 국소 기울기를 차례대로 곱한다.
dL_dz1 = dL_dprediction * w2.detach().item() * relu_gradient
manual_dw1 = dL_dz1 * x.item()
manual_db1 = dL_dz1

# ---위에까지가 수동---------
# ---아래부터는 자동---------

loss.backward()


print(f"loss: {loss.item():.6f}")
print(f"w2 gradient | manual={manual_dw2:.6f}, autograd={w2.grad.item():.6f}")
print(f"b2 gradient | manual={manual_db2:.6f}, autograd={b2.grad.item():.6f}")
print(f"w1 gradient | manual={manual_dw1:.6f}, autograd={w1.grad.item():.6f}")
print(f"b1 gradient | manual={manual_db1:.6f}, autograd={b1.grad.item():.6f}")


prediction: 1.080000, h: 1.100000, z1: 1.100000
loss: 0.006400
w2 gradient | manual=0.176000, autograd=0.176000
b2 gradient | manual=0.160000, autograd=0.160000
w1 gradient | manual=0.256000, autograd=0.256000
b1 gradient | manual=0.128000, autograd=0.128000


## 4. 실제 PyTorch 신경망의 한 번의 역전파

XOR 입력 네 개를 처리하는 작은 신경망을 만든다.
모델은 클래스 확률이 아니라 logits를 출력하므로 `BCEWithLogitsLoss`를 사용한다.

모델 구조는 `입력 2개 → 은닉 노드 4개 → 출력 1개`이다. `Tanh`는 은닉층에 비선형성을 추가해 XOR처럼 직선 하나로 나눌 수 없는 문제를 학습할 수 있게 한다.

In [6]:
X = torch.tensor([
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
])
y= torch.tensor([[0.0], [1.0], [1.0], [0.0]])

# nn.Sequential() 자체가 nn.Module을 상속받은 모델
# -> 별도 모델용 class 작성 없이 학습이 가능함 + 순전파
model=nn.Sequential(
    nn.Linear(2, 4), # 은닉층
    nn.Tanh(), # 활성화
    nn.Linear(4, 1), # 출력층
    # 출력층 이후 활성화 X -> logits로 반환
)

# 손실함수
criterion = nn.BCEWithLogitsLoss()

# 최적화
optimizer = optim.Adam(model.parameters(), lr=0.05)

print(model)
print('입력 X.shape', X.shape)
print('정답 y.shape', y.shape)

Sequential(
  (0): Linear(in_features=2, out_features=4, bias=True)
  (1): Tanh()
  (2): Linear(in_features=4, out_features=1, bias=True)
)
입력 X.shape torch.Size([4, 2])
정답 y.shape torch.Size([4, 1])


한 번의 학습 step을 단계별로 실행한다.

- `zero_grad()`는 이전 step의 기울기를 지운다.
- `model(X)`는 현재 파라미터로 logits를 계산한다.
- `loss.backward()`는 각 파라미터의 `.grad`를 채운다.
- `optimizer.step()`은 기울기를 이용해 파라미터를 변경한다.

기울기와 파라미터 변화량을 함께 출력해 `backward()`와 `step()`의 역할을 구분한다.

In [7]:
# step 전 파라미터를 clone해 optimizer.step 이후의 변화량과 비교한다.
# detach는 현재 Tensor를 autograd 계산 그래프에서 분리한다.
before_step = {
    name: parameter.detach().clone()
    for name, parameter in model.named_parameters()
}

# zero_grad는 이전 step에서 누적된 기울기를 지운다.
optimizer.zero_grad()

# 순전파: 현재 파라미터로 XOR 입력의 logits를 계산한다.
logits = model(X)

# 손실 계산: logits와 이진 정답의 차이를 하나의 loss로 만든다.
loss = criterion(logits, y)

# 역전파: 각 파라미터가 loss에 미치는 변화율을 .grad에 계산한다.
# 이 단계에서는 아직 weight와 bias가 변경되지 않는다.
loss.backward()

print("loss:", loss.item()) # loss.item()은 원소가 하나인 Tensor에서 숫자만 꺼내 Python의 일반 숫자러 변환됨

# gradient norm은 각 파라미터 Tensor에 저장된 기울기의 전체 크기를 요약한다.
for name, parameter in model.named_parameters():
    print(f"{name} gradient norm: {parameter.grad.norm().item():.6f}")

# step은 Adam 규칙과 계산된 기울기를 이용해 파라미터를 실제로 갱신한다.
optimizer.step()

# 갱신 전후 파라미터의 평균 절대 변화량을 확인한다.
for name, parameter in model.named_parameters():
    change = (parameter.detach() - before_step[name]).abs().mean().item()
    print(f"{name} mean change: {change:.6f}")

loss: 0.7143304944038391
0.weight gradient norm: 0.057816
0.bias gradient norm: 0.088402
2.weight gradient norm: 0.069411
2.bias gradient norm: 0.098526
0.weight mean change: 0.050000
0.bias mean change: 0.050000
2.weight mean change: 0.050000
2.bias mean change: 0.050000


모든 학습 파라미터에 기울기가 존재하면 손실에서 각 계층까지 역전파 경로가 연결된 것이다. `optimizer.step()` 이후 파라미터 변화량이 0보다 크면 실제 갱신도 수행된 것이다.

기울기 norm이 크다고 가중치 자체가 크다는 뜻은 아니다. 기울기는 현재 파라미터가 손실에 얼마나 영향을 주는지 나타내는 변화율이다.

## 5. 기울기 누적

PyTorch의 기울기는 기본적으로 덮어쓰지 않고 `.grad`에 더해진다. 같은 손실로 `backward()`를 두 번 실행하면 기울기 크기도 약 두 배가 된다. 일반적인 학습 루프에서 매 step마다 `zero_grad()`를 호출하는 이유이다.

In [8]:
optimizer.zero_grad()


criterion(model(X), y).backward()
first_norm = model[0].weight.grad.norm().item()


criterion(model(X), y).backward()
second_norm = model[0].weight.grad.norm().item()

print("한 번 backward한 기울기 norm:", first_norm)
print("두 번 누적된 기울기 norm:", second_norm)


optimizer.zero_grad()


한 번 backward한 기울기 norm: 0.02367822267115116
두 번 누적된 기울기 norm: 0.04735644534230232


## 6. 반복 학습과 XOR 예측

한 번의 역전파는 현재 파라미터에서의 기울기만 계산한다. 같은 학습 단계를 반복하면서 손실이 줄어드는 방향으로 파라미터를 계속 갱신해야 모델이 XOR 규칙을 학습한다.

학습 중에는 logits를 `BCEWithLogitsLoss`에 그대로 전달한다. 학습이 끝난 뒤 예측할 때만 Sigmoid로 확률을 만들고 0.5를 임계값으로 클래스를 결정한다.

In [9]:
torch.manual_seed(42)


model = nn.Sequential(
    nn.Linear(2, 4),
    nn.Tanh(),
    nn.Linear(4, 1),
)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.05)


for epoch in range(1001): # 1001번 반복

    optimizer.zero_grad()


    logits = model(X)
    loss = criterion(logits, y)


    loss.backward()
    optimizer.step()


    if epoch % 200 == 0:
        print(f"Epoch {epoch:4d} | Loss {loss.item():.6f}")


Epoch    0 | Loss 0.759778
Epoch  200 | Loss 0.008929
Epoch  400 | Loss 0.003199
Epoch  600 | Loss 0.001719
Epoch  800 | Loss 0.001092
Epoch 1000 | Loss 0.000761


In [10]:
model.eval()  # 평가 모드


with torch.no_grad():

    logits = model(X)
    probabilities = torch.sigmoid(logits)


    predictions = (probabilities >= 0.5).int()


for inputs, probability, prediction, target in zip(
    X, probabilities, predictions, y
):
    print(
        f"Input: {inputs.tolist()}, "
        f"Probability: {probability.item():.4f}, "
        f"Predicted: {prediction.item()}, "
        f"Actual: {int(target.item())}"
    )


Input: [0.0, 0.0], Probability: 0.0000, Predicted: 0, Actual: 0
Input: [0.0, 1.0], Probability: 0.9992, Predicted: 1, Actual: 1
Input: [1.0, 0.0], Probability: 0.9992, Predicted: 1, Actual: 1
Input: [1.0, 1.0], Probability: 0.0014, Predicted: 0, Actual: 0


## 핵심 정리

- 순전파는 입력에서 예측과 손실을 계산하는 과정이다.
- 역전파는 손실에서 각 파라미터까지의 기울기를 계산하는 과정이다.
- `loss.backward()`는 가중치를 변경하지 않는다.
- `optimizer.step()`에서 가중치와 편향이 실제로 변경된다.
- PyTorch 기울기는 누적되므로 일반적인 학습 step마다 `zero_grad()`가 필요하다.
- `BCEWithLogitsLoss`와 `CrossEntropyLoss`에는 활성화 전 logits를 전달한다.